In [2]:
# Procesador de Contexto Conversacional Integrado
# ====================================================

# Importar bibliotecas necesarias
import json
import logging
from typing import Dict, List, Optional, Any
from dataclasses import dataclass
import random
import os
from pathlib import Path
from typing import Tuple

# Configuración de logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

@dataclass
class DomainContext:
    """Clase para almacenar el contexto del dominio actual"""
    type: str                    # Tipo de conversación
    stage: str                   # Etapa de la conversación
    mood: str                    # Estado de ánimo del asistente
    personality: str             # Personalidad del asistente
    method: str                  # Método de interacción
    area: str                    # Área específica
    specific_topic: str          # Tema específico dentro del área
    attributes: Dict[str, Any]   # Atributos adicionales
    prompts: Dict[str, str]      # Templates de prompts
    instructions: List[str]      # Instrucciones específicas

class NaturalContextProcessor:
    """
    Procesa el contexto del dominio para generar descripciones naturales y fluidas.
    Maneja la traducción de estructuras jerárquicas a lenguaje natural.
    """
    
    def __init__(self):
        self.hierarchy_connectors = {
            'domain': {
                'start': "nos encontramos en un contexto de",
                'connect': "dentro del ámbito de"
            },
            'area': {
                'start': "el usuario busca ayuda en el área de",
                'connect': "relacionado con el área de"
            },
            'method': {
                'start': "a través de",
                'connect': "utilizando"
            },
            'topic': {
                'start': "para abordar",
                'connect': "enfocándonos en"
            },
            'subtopic': {
                'start': "específicamente en",
                'connect': "centrándonos en"
            }
        }
        
        self.node_type_descriptors = {
            'apoyo_emocional': 'apoyo emocional',
            'asistencia_practica': 'asistencia práctica',
            'analisis_informativo': 'análisis informativo',
            'clarificacion_contextual': 'clarificación contextual',
            'mediante_reflexion': 'reflexión',
            'mediante_conversacion': 'conversación',
            'mediante_explicacion': 'explicación',
            'mediante_analisis': 'análisis',
            'mediante_investigacion': 'investigación'
        }

    def process_domain_context(self, domain_type: str, context_structure: dict) -> str:
        """Procesa la estructura del contexto para generar una descripción natural."""
        try:
            # Iniciar con el tipo de dominio
            parts = [
                f"{self.hierarchy_connectors['domain']['start']} "
                f"{self.node_type_descriptors.get(domain_type, domain_type.replace('_', ' '))}"
            ]
            
            # Procesar la jerarquía del contexto
            current_level = 'area'
            for key, value in self._traverse_context(context_structure):
                connector = self._get_connector(current_level, len(parts) <= 1)
                formatted_key = self._format_node_key(key)
                parts.append(f"{connector} {formatted_key}")
                current_level = self._get_next_level(current_level)
            
            return ", ".join(parts)
        except Exception as e:
            logger.error(f"Error procesando contexto: {str(e)}")
            return f"contexto de {domain_type}"

    def _traverse_context(self, context: dict):
        """Recorre la estructura del contexto de manera recursiva."""
        for key, value in context.items():
            yield key, value
            if isinstance(value, dict):
                yield from self._traverse_context(value)

    def _get_connector(self, level: str, is_first: bool) -> str:
        """Obtiene el conector apropiado según el nivel y posición."""
        connectors = self.hierarchy_connectors.get(level, {})
        return connectors['start'] if is_first else connectors['connect']

    def _format_node_key(self, key: str) -> str:
        """Formatea la clave del nodo para presentación natural."""
        return self.node_type_descriptors.get(key, key.replace('_', ' '))

    def _get_next_level(self, current_level: str) -> str:
        """Determina el siguiente nivel en la jerarquía."""
        levels = ['domain', 'area', 'method', 'topic', 'subtopic']
        try:
            current_index = levels.index(current_level)
            return levels[current_index + 1] if current_index < len(levels) - 1 else 'subtopic'
        except ValueError:
            return 'subtopic'

class PromptManager:
    """
    Maneja la generación y formato de prompts para diferentes componentes
    del sistema de conversación.
    """
    
    def __init__(self):
        self.context_processor = NaturalContextProcessor()

    def generate_initial_prompt(self, context: DomainContext) -> str:
        """Genera el prompt inicial basado en templates del usuario."""
        try:
            user_templates = context.prompts.get('user_templates', [])
            if not user_templates:
                return f"¿Qué necesitas saber sobre {context.specific_topic}?"
            
            template = random.choice(user_templates)
            topic = (f"{context.specific_topic} - {context.attributes['category']}" 
                    if 'category' in context.attributes 
                    else context.specific_topic)
            
            return template.format(
                tema=topic,
                concepto=topic,
                situacion=topic,
                aspecto=topic
            )
        except Exception as e:
            logger.error(f"Error generando prompt inicial: {str(e)}")
            return "¿En qué puedo ayudarte?"

    def generate_domain_prompt(self, context: DomainContext) -> str:
        """Genera el prompt de dominio con contexto narrativo."""
        try:
            # Construir estructura de contexto para procesamiento
            context_structure = {
                context.area: {
                    context.method: {
                        context.specific_topic: context.attributes.get('method_context', {})
                    }
                }
            }
            
            context_description = self.context_processor.process_domain_context(
                context.type, context_structure
            )
            
            return f"""
Contexto de la Conversación:
{context_description}

Esta interacción forma parte de un proceso de {context.type.replace('_', ' ')}
que se encuentra en su etapa de {context.stage}, donde buscamos facilitar
un proceso de {context.method.replace('mediante_', '')} efectivo.

Objetivo:
Proporcionar una experiencia de {context.method.replace('mediante_', '')}
adaptada al contexto específico de {context.specific_topic},
manteniendo el enfoque en las necesidades particulares del usuario.
""".strip()

        except Exception as e:
            logger.error(f"Error generando prompt de dominio: {str(e)}")
            return ""

    def generate_assistant_prompt(self, context: DomainContext) -> str:
        """Genera el prompt del asistente con instrucciones del sistema."""
        try:
            system_base = context.prompts['system_base'].format(
                mood=context.mood,
                personality=context.personality
            )
            
            instructions = "\n".join(f"- {inst}" for inst in context.instructions)
            
            return f"""
{system_base}

Instrucciones Específicas:
{instructions}

Parámetros de Interacción:
- Tono: {context.mood.strip(', ') if context.mood else 'neutral'}
- Estilo: {context.personality.strip(' y ') if context.personality else 'directo'}
- Etapa: {context.stage}
""".strip()

        except Exception as e:
            logger.error(f"Error generando prompt del asistente: {str(e)}")
            return ""

class EnhancedConversationDomainManager:
    """
    Versión mejorada del gestor de dominios de conversación que integra
    el procesamiento natural de contexto y la generación de prompts.
    """
    
    def __init__(self, config_path: str = 'config'):
        self.config_path = Path(config_path)
        self.load_configurations()
        self.prompt_manager = PromptManager()
        self.current_context = None

    def load_configurations(self):
        """Carga todos los archivos de configuración necesarios."""
        try:
            self.types_config = self._load_json('conversation_types.json')
            self.stages_config = self._load_json('conversation_stages.json')
            self.global_config = self._load_json('global_parameters.json')
            self.extended_config = self._load_json('conversation_types-v3.json')
            logger.info("Configuraciones cargadas exitosamente")
        except Exception as e:
            logger.error(f"Error cargando configuraciones: {str(e)}")
            raise

    def _load_json(self, filename: str) -> dict:
        """Carga un archivo JSON de configuración."""
        file_path = self.config_path / filename
        with open(file_path, 'r', encoding='utf-8') as f:
            return json.load(f)

    def set_initial_domain(self) -> Optional[DomainContext]:
        """Establece el dominio inicial de la conversación."""
        try:
            # Selección aleatoria de componentes
            domain_type = random.choice(list(self.extended_config['tipos'].keys()))
            stage = random.choice(list(self.stages_config['stages'].keys()))
            mood = random.choice(self.global_config['moods'])
            personality = random.choice(self.global_config['personalities'])
            
            # Explorar jerarquía del dominio
            area, method, topic, category = self._explore_domain_hierarchy(domain_type)
            
            # Obtener configuraciones específicas
            instructions = self._get_method_instructions(domain_type, method)
            prompt_templates = self._get_prompt_templates(domain_type, method)
            
            # Crear y establecer el contexto
            self.current_context = DomainContext(
                type=domain_type,
                stage=stage,
                mood=mood,
                personality=personality,
                method=method,
                area=area,
                specific_topic=topic,
                attributes={
                    'category': category,
                    'stage_config': self.stages_config['stages'][stage],
                    'method_context': self.extended_config['tipos'][domain_type]['contexto'][area][method]
                },
                prompts=prompt_templates,
                instructions=instructions
            )
            
            return self.current_context
            
        except Exception as e:
            logger.error(f"Error estableciendo dominio inicial: {str(e)}")
            return None

    def generate_conversation_context(self) -> Dict[str, str]:
        """Genera todos los componentes del contexto conversacional."""
        if not self.current_context:
            raise ValueError("No hay un contexto de dominio establecido")
            
        return {
            'initial_prompt': self.prompt_manager.generate_initial_prompt(self.current_context),
            'domain_prompt': self.prompt_manager.generate_domain_prompt(self.current_context),
            'assistant_prompt': self.prompt_manager.generate_assistant_prompt(self.current_context)
        }
    
    def _explore_domain_hierarchy(self, domain_type: str) -> Tuple[str, str, str, str]:
        """
        Explora la jerarquía del dominio y selecciona una ruta válida
        """
        try:
            domain_context = self.extended_config['tipos'][domain_type]['contexto']
            
            # Seleccionar área
            area = random.choice(list(domain_context.keys()))
            area_context = domain_context[area]
            
            # Seleccionar método
            method = random.choice([k for k in area_context.keys() if k.startswith('mediante_')])
            method_context = area_context[method]
            
            # Seleccionar tema específico
            topic = random.choice(list(method_context.keys()))
            topic_context = method_context[topic]
            
            # Seleccionar categoría si existe
            category = ""
            if isinstance(topic_context, dict):
                category = random.choice(list(topic_context.keys()))
            
            return area, method, topic, category
            
        except Exception as e:
            logger.error(f"Error explorando jerarquía de dominio: {str(e)}")
            raise

    def _get_method_instructions(self, domain_type: str, method: str) -> List[str]:
        """
        Obtiene las instrucciones específicas del método
        """
        try:
            prompts_config = self.extended_config['tipos'][domain_type]['prompts']
            system_config = prompts_config['sistema']
            
            # Obtener instrucciones generales
            general_instructions = system_config['instrucciones'].get('general', [])
            
            # Obtener instrucciones específicas del método
            method_instructions = system_config['instrucciones'].get('por_medio', {}).get(method, [])
            
            return general_instructions + method_instructions
            
        except Exception as e:
            logger.error(f"Error obteniendo instrucciones del método: {str(e)}")
            return []

    def _get_prompt_templates(self, domain_type: str, method: str) -> Dict[str, Any]:
        """
        Obtiene los templates de prompts para el dominio y método
        """
        try:
            prompts_config = self.extended_config['tipos'][domain_type]['prompts']
            
            templates = {
                'system_base': prompts_config['sistema']['base'],
                'user_templates': []
            }
            
            # Obtener patrones de inicio para el método
            if 'usuario' in prompts_config:
                user_config = prompts_config['usuario']
                if 'patrones_inicio' in user_config:
                    templates['user_templates'] = user_config['patrones_inicio'].get(method, [])
                    
            return templates
            
        except Exception as e:
            logger.error(f"Error obteniendo templates de prompts: {str(e)}")
            return {}

    def validate_domain_context(self, context: DomainContext) -> bool:
        """
        Valida que el contexto de dominio sea coherente y completo
        """
        try:
            # Validar existencia en configuración extendida
            if context.type not in self.extended_config['tipos']:
                logger.warning(f"Tipo no válido en config extendida: {context.type}")
                return False
            
            # Validar jerarquía
            domain_context = self.extended_config['tipos'][context.type]['contexto']
            if context.area not in domain_context:
                logger.warning(f"Área no válida: {context.area}")
                return False
                
            area_context = domain_context[context.area]
            if context.method not in area_context:
                logger.warning(f"Método no válido: {context.method}")
                return False
                
            # Validar que tenga prompts e instrucciones
            if not context.prompts or not context.instructions:
                logger.warning("Faltan prompts o instrucciones")
                return False
                
            return True
            
        except Exception as e:
            logger.error(f"Error en validación de contexto: {str(e)}")
            return False

def print_conversation_context(context_dict: Dict[str, str]):
    """Imprime el contexto de la conversación de manera formateada."""
    print("\n=== Prompt Inicial (Pregunta del Usuario) ===")
    print(context_dict['initial_prompt'])
    
    print("\n=== Prompt de Dominio (Contexto Narrativo) ===")
    print(context_dict['domain_prompt'])
    
    print("\n=== Prompt del Asistente (Instrucciones) ===")
    print(context_dict['assistant_prompt'])

# Ejemplo de uso
if __name__ == "__main__":
    try:
        # Inicializar el gestor de dominio mejorado
        domain_manager = EnhancedConversationDomainManager()
        
        # Establecer el dominio inicial
        if domain_manager.set_initial_domain():
            # Generar y mostrar el contexto de la conversación
            conversation_context = domain_manager.generate_conversation_context()
            print_conversation_context(conversation_context)
        else:
            print("Error: No se pudo establecer el dominio inicial")
            
    except Exception as e:
        logger.error(f"Error en la ejecución principal: {str(e)}")
        raise

2024-11-22 19:28:48,331 - INFO - Configuraciones cargadas exitosamente



=== Prompt Inicial (Pregunta del Usuario) ===
¿Cómo puedo analizar metodos_estudio - organizacion?

=== Prompt de Dominio (Contexto Narrativo) ===
Contexto de la Conversación:
nos encontramos en un contexto de análisis informativo, el usuario busca ayuda en el área de educativo, utilizando análisis, enfocándonos en metodos estudio, centrándonos en metodos estudio, centrándonos en organizacion, centrándonos en comprension

Esta interacción forma parte de un proceso de analisis informativo
que se encuentra en su etapa de consolidation, donde buscamos facilitar
un proceso de analisis efectivo.

Objetivo:
Proporcionar una experiencia de analisis
adaptada al contexto específico de metodos_estudio,
manteniendo el enfoque en las necesidades particulares del usuario.

=== Prompt del Asistente (Instrucciones) ===
Eres un asistente especializado en análisis y exploración de conocimiento, manteniendo un tono neutro y comunicándote de manera concisa.

Instrucciones Específicas:
- Estructura la in